# PPO TL;DR recipe — verification notebook

Companion to [`README.md`](./README.md). The README documents the SFT → reward model → PPO commands; this notebook **runs the cheap parts** (tokenizer inspection, reward-model diagnostics on a published checkpoint) so you can see real numbers before committing GPU time to the full pipeline.

What this notebook does:
- Inspects the tokenizer configs of the published `trl-lib/pythia-1b-deduped-tldr-{sft,rm}` checkpoints and reports any inconsistency (relevant to the original concern in [issue #2015](https://github.com/huggingface/trl/issues/2015)).
- Runs [`evaluate_reward_model.py`](./evaluate_reward_model.py) on a small sample of `trl-lib/tldr-preference` to produce real diagnostic numbers (chosen/rejected accuracy, margin distribution, length-bias correlation, failure CSV).

What this notebook does NOT do:
- It does **not** retrain the SFT or reward model from scratch. Those steps need GPU compute the typical laptop does not have. See README Steps 1–3 for the commands; run them on Colab Pro / Paperspace / a cluster.
- It does **not** validate end-to-end PPO convergence. Same reason. For a smaller-scale pipeline sanity check use `bash examples/scripts/ppo/smoke_test.sh`.

## 0. Environment

In [1]:
import os, sys, json, subprocess

import torch
from transformers import AutoTokenizer

print('torch:', torch.__version__)
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print('device:', device)

torch: 2.10.0
device: mps


## 1. Tokenizer inspection of the published checkpoints

The original [issue #2015](https://github.com/huggingface/trl/issues/2015) (second comment) flagged that the legacy `cleanrl/EleutherAI_pythia-1b-deduped__reward__tldr` checkpoint has no `pad_token`, inconsistent with the SFT checkpoint. Let's check the situation on the maintained TRL pair (`trl-lib/pythia-1b-deduped-tldr-{sft,rm}`).

In [2]:
sft_tok = AutoTokenizer.from_pretrained('trl-lib/pythia-1b-deduped-tldr-sft')
rm_tok  = AutoTokenizer.from_pretrained('trl-lib/pythia-1b-deduped-tldr-rm')

print('SFT pad_token:', repr(sft_tok.pad_token), 'id:', sft_tok.pad_token_id)
print('RM  pad_token:', repr(rm_tok.pad_token),  'id:', rm_tok.pad_token_id)
print('SFT eos_token:', repr(sft_tok.eos_token), 'id:', sft_tok.eos_token_id)
print('RM  eos_token:', repr(rm_tok.eos_token),  'id:', rm_tok.eos_token_id)

SFT pad_token: '[PAD]' id: 50277
RM  pad_token: None id: None
SFT eos_token: '<|endoftext|>' id: 0
RM  eos_token: '<|endoftext|>' id: 0


### What you should see

Empirically (as of 2026-05):
```
SFT pad_token: '[PAD]'        id: 50277
RM  pad_token: None           id: None
SFT eos_token: '<|endoftext|>' id: 0
RM  eos_token: '<|endoftext|>' id: 0
```
**The published `trl-lib/pythia-1b-deduped-tldr-rm` checkpoint also lacks a saved `pad_token`** — same metadata gap as the cleanrl checkpoints originally flagged in #2015. This is a quirk of how the checkpoint was produced, *not* a property of the recipe.

Why this is not a blocker for using the checkpoint for PPO/RLOO:
- At inference time, both [`RewardTrainer`](../../../trl/trainer/reward_trainer.py) (line ~471) and [`SFTTrainer`](../../../trl/trainer/sft_trainer.py) (line ~1157) do `pad_token = args.pad_token or processing_class.pad_token or processing_class.eos_token` — i.e. they fall through to `eos_token` if `pad_token` is unset. PPO's data path likewise pads with the policy tokenizer (which loads from `--model_name_or_path EleutherAI/pythia-1b-deduped` and gets `[PAD]` added on script init, see [`ppo_tldr.py`](./ppo_tldr.py) line 99).
- The reward model is used as a *scorer*, not as a generator. It receives the policy's already-tokenized rollouts and produces a scalar; the RM tokenizer's `pad_token` isn't on the critical path.

Why the README's recipe avoids the gap when you train fresh: Step 2 (`reward_modeling.py`) loads its base tokenizer from the SFT output dir of Step 1. The SFT output already has `pad_token='[PAD]'` set (via the fall-through logic above and the explicit `tokenizer.add_special_tokens({'pad_token': '[PAD]'})` pattern used in `ppo_tldr.py`). So Step 2's saved RM checkpoint inherits that pad_token. The mismatch only persists on the legacy published artifacts.

## 2. Reward-model diagnostics on the published checkpoint

Runs [`evaluate_reward_model.py`](./evaluate_reward_model.py) on a sample of `trl-lib/tldr-preference`. The script itself sets `tokenizer.pad_token = tokenizer.eos_token` if the loaded reward model has no pad_token, mirroring the fall-through logic inside the trainers.

On a laptop, 50-200 pairs is a reasonable sample. Tune `NUM_SAMPLES` for your time budget.

In [3]:
NUM_SAMPLES = 200
OUTPUT_DIR = 'reward_model_eval'

# When this notebook is executed, Jupyter sets CWD to the notebook's directory
# (examples/scripts/ppo/), so the script is a same-directory relative path.
SCRIPT = 'evaluate_reward_model.py'

cmd = [
    sys.executable, SCRIPT,
    '--reward_model_path', 'trl-lib/pythia-1b-deduped-tldr-rm',
    '--dataset_name', 'trl-lib/tldr-preference',
    '--split', 'validation',
    '--num_samples', str(NUM_SAMPLES),
    '--output_dir', OUTPUT_DIR,
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:\n', result.stderr[-2000:])
    raise SystemExit(result.returncode)

Running: /Users/kohsheentiku/Desktop/Open-source/trl/.venv/bin/python evaluate_reward_model.py --reward_model_path trl-lib/pythia-1b-deduped-tldr-rm --dataset_name trl-lib/tldr-preference --split validation --num_samples 200 --output_dir reward_model_eval
Using device: mps
Loading tokenizer + reward model from: trl-lib/pythia-1b-deduped-tldr-rm
Loading dataset: trl-lib/tldr-preference [split=validation]
Evaluating 200 pairs
  [50/200] running accuracy=0.660, mean margin=0.607
  [100/200] running accuracy=0.660, mean margin=0.581
  [150/200] running accuracy=0.673, mean margin=0.559
  [200/200] running accuracy=0.670, mean margin=0.594
Wrote summary: reward_model_eval/summary.json
Wrote failures (98 rows): reward_model_eval/failures.csv

=== Reward model diagnostic summary ===
  reward_model_path: trl-lib/pythia-1b-deduped-tldr-rm
  dataset_name: trl-lib/tldr-preference
  split: validation
  n_pairs: 200
  accuracy: 0.6700
  mean_reward_chosen: 3.6995
  mean_reward_rejected: 3.1058
  me

In [4]:
with open(os.path.join(OUTPUT_DIR, 'summary.json')) as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

{
  "reward_model_path": "trl-lib/pythia-1b-deduped-tldr-rm",
  "dataset_name": "trl-lib/tldr-preference",
  "split": "validation",
  "n_pairs": 200,
  "accuracy": 0.67,
  "mean_reward_chosen": 3.699526339173317,
  "mean_reward_rejected": 3.105778040289879,
  "mean_margin": 0.5937482988834382,
  "median_margin": 0.6108624935150146,
  "min_margin": -3.3307085037231445,
  "max_margin": 4.4827200174331665,
  "mispreferred_fraction": 0.33,
  "near_zero_fraction": 0.285,
  "margin_threshold": 0.5,
  "length_bias_pearson": 0.31194381214997235,
  "mean_chosen_length_tokens": 34.955,
  "mean_rejected_length_tokens": 31.94
}


In [5]:
import csv

rows = []
with open(os.path.join(OUTPUT_DIR, 'failures.csv')) as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f'{len(rows)} flagged pairs (mispreferred or |margin| < threshold)')
for r in rows[:5]:
    print(f"  idx={r['index']} cat={r['category']} margin={float(r['margin']):.3f} len_c={r['len_chosen']} len_r={r['len_rejected']}")
    print(f"     prompt: {r['prompt'][:120]!r}")
    print()

98 flagged pairs (mispreferred or |margin| < threshold)
  idx=3 cat=mispreferred margin=-0.597 len_c=33 len_r=40
     prompt: "SUBREDDIT: r/legaladvice\n\nTITLE: My employer processed my time-sheets late because of their Christmas break. Now I'm not"

  idx=6 cat=mispreferred margin=-0.613 len_c=25 len_r=39
     prompt: 'SUBREDDIT: r/relationships\n\nTITLE: Me [21F] going to a small festival at which my recent ex-boyfriend [21M] will also be'

  idx=8 cat=near_zero margin=0.207 len_c=40 len_r=32
     prompt: 'SUBREDDIT: r/Advice\n\nTITLE: How to recover for an all nighter, and prepare for another one very soon?\n\nPOST: I have a ve'

  idx=9 cat=mispreferred margin=-0.381 len_c=31 len_r=41
     prompt: 'SUBREDDIT: r/relationships\n\nTITLE: Me [27 F] with my Dad [60 M], emailed me and my siblings this morning informing me th'

  idx=11 cat=near_zero margin=0.284 len_c=36 len_r=20
     prompt: 'TITLE: Sixteen writers defend their decision to not have kids in book edited by LA based Megha

## 3. Interpretation

A healthy reward model on TL;DR-preference should show:
- `accuracy` meaningfully above 0.5.
- `mean_margin` positive.
- `length_bias_pearson` close to zero. A strongly positive value (say > 0.4) means the model is partly rewarding length rather than quality — a known failure mode of preference reward models.
- `near_zero_fraction` not dominant. A model where most pairs sit near zero margin gives PPO almost no signal.

If any of those look off on **your** trained reward model (Step 2 of the recipe), don't proceed to PPO until you've debugged. The `failures.csv` is the place to start — look for systematic patterns (very short rejected vs. very long chosen, refusal-style chosen, etc.).